In [1]:
import os
import json
import pandas as pd
import traceback

In [2]:
from langchain.chat_models import ChatOpenAI

In [3]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

True

In [6]:
KEY=os.getenv("OPENAI_API_KEY")
KEY

'sk-proj-xOp2svyhOMJQH9k1RzjeT3BlbkFJpjgY2YmiumpX8jRKJ3bU'

In [109]:
llm=ChatOpenAI(openai_api_key=KEY, model_name="gpt-3.5-turbo", temperature=0.15)

In [110]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [111]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [112]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [113]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [114]:
quiz_chain=LLMChain(llm=llm,prompt=quiz_generation_prompt,output_key="quiz",verbose=True)

In [115]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [116]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject","quiz"], template=TEMPLATE)

In [117]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [118]:
generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [119]:
file_path=r"/workspaces/mcqgen/data.txt"
file_path

'/workspaces/mcqgen/data.txt'

In [120]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [121]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [122]:
NUMBER=15 
SUBJECT="Galaxy Z Flip6"
TONE="Medium"

In [123]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text:What's new and different about the <Galaxy Z Flip6> ? 	"Would you believe it if I told you there's a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!

The Z Flip6's 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.
Additionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy

In [124]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost:.3f}")

Total Tokens:5414
Prompt Tokens:3232
Completion Tokens:2182
Total Cost:0.009


In [125]:
response

{'text': 'What\'s new and different about the <Galaxy Z Flip6> ? \t"Would you believe it if I told you there\'s a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!\n\nThe Z Flip6\'s 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.\nAdditionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy more vivid LED effects on the screen with interactive motion content by attaching LED Effe

In [126]:
quiz = response.get('quiz')
quiz = json.loads(quiz)

In [127]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [128]:
quiz_table_data

[{'MCQ': 'What is a key design element of the Galaxy Z Flip6?',
  'Choices': 'a: Improved camera performance | b: Enhanced battery life | c: Exclusive color options | d: Advanced processor',
  'Correct': 'a'},
 {'MCQ': 'What material is used for the Galaxy Z Flip6 cases?',
  'Choices': 'a: Recycled plastic | b: Recycled glass | c: Recycled aluminum | d: Recycled cobalt',
  'Correct': 'a'},
 {'MCQ': 'What is the largest battery ever in the Galaxy Z Flip6?',
  'Choices': 'a: 3500mAh | b: 4000mAh | c: 4500mAh | d: 5000mAh',
  'Correct': 'b'},
 {'MCQ': 'Which feature of the Galaxy Z Flip6 helps in translating conversations?',
  'Choices': 'a: Interpreter | b: Chat Assist | c: FlexWindow | d: Zoom Rocker',
  'Correct': 'a'},
 {'MCQ': 'What is the new premium Flip design of the Galaxy Z Flip6 known for?',
  'Choices': 'a: Improved durability | b: Enhanced camera performance | c: Vibrant color options | d: Advanced processor',
  'Correct': 'a'},
 {'MCQ': 'What does the Camcorder grip on the G

In [129]:
quiz = pd.DataFrame(quiz_table_data)

In [130]:
quiz.to_csv("GalaxyZFlip6.csv",index=False)